In [2]:
import langchain

### Introduction to Data Ingestion 

In [3]:
import os
from typing import List, Dict, Any
import pandas as pd


In [4]:
from langchain_core.documents import Document
from langchain_text_splitters import(
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print("Set up completed!")


Set up completed!


### Understanding Document Structure In Langchain

In [5]:
doc = Document(
    page_content = "This is the content that will be embedded and searched",
    metadata = {
        "source" : "example.txt",
        "page" : 1,
        "author" : "Aakash Gupta",
        "date_created" : "2026-08-28",
        "custom_field" : "any_value"
    }
)

print("Document Structure")

print(f"Content : {doc.page_content}")
print(f"Metadata : {doc.metadata}")

# Why metadata matters:
print("\n Metadata is crucial for:")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing context in responses")
print("- Debugging and auditing")


Document Structure
Content : This is the content that will be embedded and searched
Metadata : {'source': 'example.txt', 'page': 1, 'author': 'Aakash Gupta', 'date_created': '2026-08-28', 'custom_field': 'any_value'}

 Metadata is crucial for:
- Filtering search results
- Tracking document sources
- Providing context in responses
- Debugging and auditing


In [6]:
type(doc)

langchain_core.documents.base.Document

### Text Files(.txt) - The Simplest Case (#2-text-files)

In [7]:
# create a simple text file

import os
os.makedirs("data/text_files", exist_ok = True)

In [8]:
sample_texts = {
    "data/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, general-purpose programming language designed with an emphasis on code readability, simplicity, and ease of use. 
Created by Guido van Rossum and first released in 1991, it has become one of the world's most popular languages, powering fields like artificial intelligence, web development, data science, and automation.

Key features: 
- Easy
- Fast
- Used for AI
- User friendly

This is the end of Python.""",

    "data/text_files/machine_learning.txt": """Machine Learning Introduction
Machine learning is a subset of Artificial Intelligence that uses data and algorithms to let computers train themselves 
and solve problems without being explicitly programmed with every single rule.

Key features:
- Data-Driven Learning
- Pattern Recognition
- Process Automation
- Adaptability

This is the end of Machine learning."""
}

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding = "utf-8") as f:
        f.write(content)

print("Sample text files created !")

Sample text files created !


### TextLoader - Read Single File

In [9]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/text_files/python_intro.txt", encoding="utf-8")

documents = loader.load()
print(type(documents))
print(documents)


<class 'list'>
[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content="Python Programming Introduction\n\nPython is a high-level, general-purpose programming language designed with an emphasis on code readability, simplicity, and ease of use. \nCreated by Guido van Rossum and first released in 1991, it has become one of the world's most popular languages, powering fields like artificial intelligence, web development, data science, and automation.\n\nKey features: \n- Easy\n- Fast\n- Used for AI\n- User friendly\n\nThis is the end of Python.")]


C:\Users\ARTI\AppData\Local\Temp\ipykernel_9804\3237068334.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [10]:
print(f"Loaded {len(documents)} document")
print(f"Content preview: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")


Loaded 1 document
Content preview: Python Programming Introduction

Python is a high-level, general-purpose programming language design...
Metadata: {'source': 'data/text_files/python_intro.txt'}


### DirectoryLoader - Multiple Text Files

In [12]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Load all the text files from the directory
dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt",                 # Pattern to match files
    loader_cls=TextLoader,           # Loader class to use
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)

documents = dir_loader.load()

print(f"Loaded {len(documents)} documents")

for i, doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print(f"Source: {doc.metadata['source']}")
    print(f"Length: {len(doc.page_content)} characters")
    

100%|██████████| 2/2 [00:00<00:00, 125.02it/s]

Loaded 2 documents

Document 1:
Source: data\text_files\machine_learning.txt
Length: 363 characters

Document 2:
Source: data\text_files\python_intro.txt
Length: 464 characters


In [13]:
print("📂 DirectoryLoader - Advantages")
print("-" * 40)

print("✅ Loads multiple files at once from a directory.")
print("⏱️ Saves time compared to loading files individually.")
print("🔍 Supports glob patterns to select specific files.")
print("📚 Useful for RAG projects with many documents.")
print("📁 Can search inside subdirectories.")


print("\n📂 DirectoryLoader - Disadvantages")
print("-" * 40)

print("⚠️ Can load unnecessary files if the pattern is too broad.")
print("🧠 Uses more memory when loading many/large files.")
print("📄 All files should be of the same type when using one loader.")
print("🐌 Large directories can take more time to load.")

📂 DirectoryLoader - Advantages
----------------------------------------
✅ Loads multiple files at once from a directory.
⏱️ Saves time compared to loading files individually.
🔍 Supports glob patterns to select specific files.
📚 Useful for RAG projects with many documents.
📁 Can search inside subdirectories.

📂 DirectoryLoader - Disadvantages
----------------------------------------
⚠️ Can load unnecessary files if the pattern is too broad.
🧠 Uses more memory when loading many/large files.
📄 All files should be of the same type when using one loader.
🐌 Large directories can take more time to load.


### Text Splitting Strategies

In [14]:
# Different text splitting strategies

from langchain_text_splitters import(
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

print(documents)


[Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Introduction\nMachine learning is a subset of Artificial Intelligence that uses data and algorithms to let computers train themselves \nand solve problems without being explicitly programmed with every single rule.\n\nKey features:\n- Data-Driven Learning\n- Pattern Recognition\n- Process Automation\n- Adaptability\n\nThis is the end of Machine learning.'), Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content="Python Programming Introduction\n\nPython is a high-level, general-purpose programming language designed with an emphasis on code readability, simplicity, and ease of use. \nCreated by Guido van Rossum and first released in 1991, it has become one of the world's most popular languages, powering fields like artificial intelligence, web development, data science, and automation.\n\nKey features: \n- Easy\n- Fast\n- Used for AI\n- User friendly\n\nThis 

In [15]:
# Method 1 - Character Text Splitter

text = documents[0].page_content
text

'Machine Learning Introduction\nMachine learning is a subset of Artificial Intelligence that uses data and algorithms to let computers train themselves \nand solve problems without being explicitly programmed with every single rule.\n\nKey features:\n- Data-Driven Learning\n- Pattern Recognition\n- Process Automation\n- Adaptability\n\nThis is the end of Machine learning.'

In [16]:
# Method 1 - Character-based splitting

print("Character Text Splitter :-")

char_splitter = CharacterTextSplitter(
    separator=" ",          # Split on spaces
    chunk_size=200,          # Max chunk size in characters
    chunk_overlap=20,        # Overlap between chunks
    length_function=len      # How to measure chunk size
)

char_chunks = char_splitter.split_text(text)

print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")


Character Text Splitter :-
Created 2 chunks
First chunk: Machine Learning Introduction
Machine learning is a subset of Artificial Intelligence that uses data...


In [17]:
print(char_chunks[0])
print("-" * 100)
print(char_chunks[1])

Machine Learning Introduction
Machine learning is a subset of Artificial Intelligence that uses data and algorithms to let computers train themselves 
and solve problems without being explicitly
----------------------------------------------------------------------------------------------------
being explicitly programmed with every single rule.

Key features:
- Data-Driven Learning
- Pattern Recognition
- Process Automation
- Adaptability

This is the end of Machine learning.


In [18]:
# Method 1 - Character-based splitting

print("1. Character Text Splitter :-")

char_splitter = CharacterTextSplitter(
    separator="\n",          # Split on new lines
    chunk_size=200,          # Max chunk size in characters
    chunk_overlap=20,        # Overlap between chunks
    length_function=len      # How to measure chunk size
)

char_chunks = char_splitter.split_text(text)

print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")


1. Character Text Splitter :-
Created 3 chunks
First chunk: Machine Learning Introduction
Machine learning is a subset of Artificial Intelligence that uses data...


In [19]:
print(char_chunks)

['Machine Learning Introduction\nMachine learning is a subset of Artificial Intelligence that uses data and algorithms to let computers train themselves', 'and solve problems without being explicitly programmed with every single rule.\nKey features:\n- Data-Driven Learning\n- Pattern Recognition\n- Process Automation\n- Adaptability', '- Adaptability\nThis is the end of Machine learning.']


In [20]:
# Method 2 - Recursive character splitting (Recommended)

print("\n2. RECURSIVE CHARACTER TEXT SPLITTER")

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],   # Try these separators in order
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(text)

print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}...")



2. RECURSIVE CHARACTER TEXT SPLITTER
Created 3 chunks
First chunk: Machine Learning Introduction
Machine learning is a subset of Artificial Intelligence that uses data...


In [21]:
print(recursive_chunks[0])
print("-" * 100)
print(recursive_chunks[1])
print("-" * 100)
print(recursive_chunks[2])


Machine Learning Introduction
Machine learning is a subset of Artificial Intelligence that uses data and algorithms to let computers train themselves
----------------------------------------------------------------------------------------------------
and solve problems without being explicitly programmed with every single rule.
----------------------------------------------------------------------------------------------------
Key features:
- Data-Driven Learning
- Pattern Recognition
- Process Automation
- Adaptability

This is the end of Machine learning.


In [22]:
# Create without neural break points
# Good Overlapping technique

simple_text = "This is sentence one. This is sentence two. This is sentence three. This is sentence four. This is sentence five. This is sentence six. This is sentence seven. This is sentence eight. This is sentence nine. This is sentence ten."

splitter = RecursiveCharacterTextSplitter(
    separators=[" "],       # Only split on spaces
    chunk_size=80,
    chunk_overlap=20,
    length_function=len
)

chunks = splitter.split_text(simple_text)

print(f"\nSimple text example - {len(chunks)} chunks:\n")

for i in range(len(chunks) - 1):
    print(f"Chunk {i+1}: '{chunks[i]}'")
    print(f"Chunk {i+2}: '{chunks[i+1]}'")
    print()



Simple text example - 4 chunks:

Chunk 1: 'This is sentence one. This is sentence two. This is sentence three. This is'
Chunk 2: 'three. This is sentence four. This is sentence five. This is sentence six. This'

Chunk 2: 'three. This is sentence four. This is sentence five. This is sentence six. This'
Chunk 3: 'sentence six. This is sentence seven. This is sentence eight. This is sentence'

Chunk 3: 'sentence six. This is sentence seven. This is sentence eight. This is sentence'
Chunk 4: 'This is sentence nine. This is sentence ten.'



In [23]:
# Method 3: Token-based splitting

print("\n3. TOKEN TEXT SPLITTER")

token_splitter = TokenTextSplitter(
    chunk_size=50,        # Size in tokens, not characters
    chunk_overlap=10      # Overlap in tokens
)

token_chunks = token_splitter.split_text(text)

print(f"Created {len(token_chunks)} chunks")
print(f"First chunk: {token_chunks[0][:100]}...")



3. TOKEN TEXT SPLITTER


Created 2 chunks
First chunk: Machine Learning Introduction
Machine learning is a subset of Artificial Intelligence that uses data...


In [24]:
print("📚 TEXT SPLITTING METHODS COMPARISON")
print("=" * 55)


print("\n1️⃣ CharacterTextSplitter")
print("-" * 30)
print("✅ Simple and predictable")
print("✅ Good for structured text")
print("⚠️ May break mid-sentence")
print("👉 Use when: Text has clear delimiters")


print("\n2️⃣ RecursiveCharacterTextSplitter ⭐")
print("-" * 30)
print("✅ Respects text structure")
print("✅ Tries multiple separators")
print("✅ Best general-purpose splitter")
print("⚠️ Slightly more complex")
print("👉 Use when: Default choice for most texts / RAG")


print("\n3️⃣ TokenTextSplitter")
print("-" * 30)
print("✅ Respects model token limits")
print("✅ More accurate when token count matters")
print("⚠️ Slower than character-based splitting")
print("👉 Use when: Working with token-limited models")


print("\n🧠 EASY WAY TO REMEMBER")
print("-" * 30)
print("🔤 Character  → Simple & Predictable")
print("🔄 Recursive  → Best General-Purpose ⭐")
print("🔢 Token      → Model / Token Limits")


📚 TEXT SPLITTING METHODS COMPARISON

1️⃣ CharacterTextSplitter
------------------------------
✅ Simple and predictable
✅ Good for structured text
⚠️ May break mid-sentence
👉 Use when: Text has clear delimiters

2️⃣ RecursiveCharacterTextSplitter ⭐
------------------------------
✅ Respects text structure
✅ Tries multiple separators
✅ Best general-purpose splitter
⚠️ Slightly more complex
👉 Use when: Default choice for most texts / RAG

3️⃣ TokenTextSplitter
------------------------------
✅ Respects model token limits
✅ More accurate when token count matters
⚠️ Slower than character-based splitting
👉 Use when: Working with token-limited models

🧠 EASY WAY TO REMEMBER
------------------------------
🔤 Character  → Simple & Predictable
🔄 Recursive  → Best General-Purpose ⭐
🔢 Token      → Model / Token Limits
